# Resolviendo ecuaciones con SymPy

**SymPy** es una librería de Python para matemática *simbólica*: en vez de trabajar con números decimales aproximados (como NumPy), trabaja con expresiones exactas — fracciones, raíces, símbolos como $x$ o $\pi$ — igual que lo harías a mano en papel.

En este notebook vamos a ver lo esencial para resolver ecuaciones con SymPy:

1. Instalación e importación
2. Símbolos y expresiones
3. Ecuaciones algebraicas simples con `solve()`
4. `solveset()`: la alternativa moderna (y por qué devuelve *conjuntos*)
5. Sistemas de ecuaciones lineales
6. Verificar que una solución es correcta

No cubrimos ecuaciones diferenciales ni resolución numérica aquí — este es el recorrido básico para perder el miedo a la sintaxis.

## 1. Instalación e importación

Si no tienes SymPy instalado, corre `pip install sympy` en una celda o en tu terminal (el `!` al inicio ejecuta el comando en la terminal desde el notebook).

In [1]:
# Descomenta la siguiente línea si SymPy no está instalado
# !pip install sympy

import sympy as sp
from sympy import symbols, Eq, solve, solveset, S

# init_printing hace que las expresiones se vean "bonitas" (notación matemática)
# en lugar de texto plano
sp.init_printing()

## 2. Símbolos y expresiones

A diferencia de una variable normal de Python, un **símbolo** de SymPy no tiene un valor numérico — representa una incógnita matemática, como la $x$ en una ecuación.

Se crean con `symbols()`.

In [2]:
x, y, z = symbols('x y z')

# Una expresión simbólica: no se evalúa, se manipula
expresion = x**2 - 5*x + 6
expresion

In [3]:
# SymPy puede simplificar, expandir, factorizar...
sp.factor(expresion)

## 3. Ecuaciones algebraicas simples con `solve()`

`solve()` es la función clásica y más usada de SymPy. Hay dos formas de plantear una ecuación:

- **Forma implícita**: le pasas una expresión y SymPy asume que está igualada a cero. Por ejemplo, `solve(x**2 - 5*x + 6, x)` resuelve $x^2-5x+6=0$.
- **Forma explícita con `Eq`**: usas `Eq(lado_izquierdo, lado_derecho)` cuando la ecuación no está igualada a cero, por ejemplo $2x+3=x-1$.

**Importante:** en SymPy nunca se usa `==` para plantear una ecuación (`==` compara, no define una igualdad simbólica). Para eso existe `Eq()`.

In [4]:
# Forma implícita: x**2 - 5*x + 6 = 0
soluciones = solve(x**2 - 5*x + 6, x)
soluciones

In [5]:
# Forma explícita con Eq: 2x + 3 = x - 1
ecuacion = Eq(2*x + 3, x - 1)
solve(ecuacion, x)

In [6]:
# También funciona con ecuaciones que no tienen solución racional "bonita"
solve(x**2 - 2, x)   # las soluciones se dejan exactas, como sqrt(2), no 1.4142...

In [7]:
# Y con parámetros simbólicos: resolver la fórmula general de la cuadrática
a, b, c = symbols('a b c')
solve(a*x**2 + b*x + c, x)   # SymPy te devuelve la fórmula general

## 4. `solveset()`: la alternativa moderna

`solve()` es cómodo, pero tiene un problema: a veces "esconde" información (por ejemplo, no siempre distingue si hay infinitas soluciones). `solveset()` es la función recomendada por los desarrolladores de SymPy para resolver una sola ecuación, porque:

- Siempre devuelve un **conjunto** matemático (`FiniteSet`, `EmptySet`, un intervalo, etc.), nunca una lista ambigua.
- Te deja especificar el **dominio** en el que buscas soluciones (reales, complejos, enteros...).

La sintaxis es: `solveset(ecuacion, variable, dominio)`.

In [8]:
# Mismo ejemplo de antes, con solveset
solveset(x**2 - 5*x + 6, x)

In [9]:
# Una ecuación trigonométrica tiene infinitas soluciones (es periódica).
# solve() luchará por darte una lista finita; solveset() te da el conjunto real:
solveset(sp.sin(x), x, domain=S.Reals)

In [10]:
# Si restringimos el dominio a los enteros, el resultado cambia:
solveset(sp.Eq(x**2, 4), x, domain=S.Reals)

## 5. Sistemas de ecuaciones lineales

Para resolver **varias ecuaciones a la vez**, le pasamos a `solve()` una lista de ecuaciones y una lista de incógnitas.

In [11]:
# Sistema 2x2:
#   x + y = 10
#   x - y = 2
eq1 = Eq(x + y, 10)
eq2 = Eq(x - y, 2)

solve([eq1, eq2], [x, y])

In [12]:
# Sistema 3x3
eq1 = Eq(x + y + z, 6)
eq2 = Eq(2*x - y + z, 3)
eq3 = Eq(x + 2*y - z, 2)

solve([eq1, eq2, eq3], [x, y, z])

También existe `linsolve()`, pensado específicamente para sistemas **lineales** (es más rápido y explícito que `solve()` para este caso, y también devuelve un conjunto):

In [13]:
from sympy import linsolve

linsolve([eq1, eq2, eq3], [x, y, z])

## 6. Verificar que una solución es correcta

Una vez que tienes una solución, es buena práctica **comprobarla** sustituyéndola de vuelta en la ecuación original con `.subs()`. Si la sustitución da cero (o simplifica a `True`), la solución es correcta.

In [14]:
expr = x**2 - 5*x + 6
raices = solve(expr, x)

for r in raices:
    resultado = expr.subs(x, r)
    print(f"x = {r}  ->  {expr} = {resultado}")

x = 2  ->  x**2 - 5*x + 6 = 0
x = 3  ->  x**2 - 5*x + 6 = 0


In [15]:
# Para una ecuación con Eq, .subs() te da una expresión booleana que puedes verificar:
ecuacion = Eq(2*x + 3, x - 1)
solucion = solve(ecuacion, x)[0]

ecuacion.subs(x, solucion)   # debe simplificar a True

## Resumen rápido

| Quiero...                                   | Uso...                                        |
|----------------------------------------------|------------------------------------------------|
| Resolver una ecuación (forma rápida)          | `solve(expr, x)` o `solve(Eq(lhs, rhs), x)`     |
| Resolver una ecuación (forma robusta/moderna) | `solveset(expr, x, domain=S.Reals)`             |
| Resolver un sistema de ecuaciones             | `solve([eq1, eq2, ...], [x, y, ...])`           |
| Resolver un sistema **lineal**                | `linsolve([eq1, eq2, ...], [x, y, ...])`        |
| Verificar una solución                        | `expr.subs(x, valor)` o `ecuacion.subs(x, valor)` |

**Para seguir explorando:** `nsolve()` (soluciones numéricas de ecuaciones que no tienen forma cerrada) y `dsolve()` (ecuaciones diferenciales) son los siguientes pasos naturales una vez que domines lo anterior — muy útiles, por ejemplo, para verificar a mano los resultados de un método de diferencias finitas.

## Ejercicios propuestos

1. Resuelve $3x^2 - 12 = 0$ con `solve()` y comprueba el resultado con `.subs()`.
2. Usa `solveset()` para encontrar, en el dominio de los reales, todas las soluciones de $\cos(x) = 0$.
3. Plantea y resuelve el sistema: $2x + 3y = 12$, $x - y = 1$.
4. Usa `solve()` con parámetros simbólicos para encontrar la fórmula que despeja $t$ de $s = s_0 + v_0 t + \tfrac{1}{2}at^2$ (hay dos soluciones — ¿por qué?).